<a href="https://colab.research.google.com/github/dwikidias/scikit-learn-Cookbook/blob/main/Bab_2_Pre_Model_Workflow_and_Data_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BAB 2: PRE-MODEL WORKFLOW AND DATA PREPROCESSING

* **Buku Referensi:** *scikit-learn Cookbook (Third Edition)* - John Sukup (Packt / O'Reilly, 2025)
* **Topik:** Data Cleaning, Handling Missing Data, Feature Scaling, Categorical Encoding, ColumnTransformer, and Feature Engineering

---

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import sklearn
print(f"Versi scikit-learn: {sklearn.__version__}")

Versi scikit-learn: 1.6.1


## 1. Dampak Data Mentah & Penanganan Nilai Hilang (Handling Missing Data)

Dalam machine learning berlaku prinsip *Garbage In, Garbage Out* (GIGO). Kualitas data masukan adalah faktor penentu paling kritis bagi performa model. Salah satu masalah data paling umum di dunia nyata adalah adanya **nilai yang hilang (*missing data / NaN*)**.

Sebagian besar algoritma dalam `scikit-learn` (seperti regresi linier, SVM, KNN, dan jaringan saraf) tidak dapat mengeksekusi data yang memiliki nilai NaN secara langsung dan akan memunculkan error jika data tidak ditangani terlebih dahulu.

### Mekanisme Terjadinya Missing Data:
1. **MCAR (*Missing Completely at Random*):** Data hilang secara acak murni tanpa pola apa pun yang berkaitan dengan variabel lain.
2. **MAR (*Missing at Random*):** Pola data hilang memiliki kaitan dengan variabel lain yang teramati dalam dataset.
3. **MNAR (*Missing Not at Random*):** Data hilang berkaitan langsung dengan nilai variabel itu sendiri (misalnya responden bergaji sangat tinggi sengaja tidak mengisi kolom gaji).

---

### Strategi Imputasi dalam scikit-learn:
1. **`SimpleImputer` (Univariate Imputation):**
   Mengganti nilai hilang pada satu fitur menggunakan nilai statistik dari fitur itu sendiri:
   - `strategy='mean'`: Cocok untuk data numerik yang terdistribusi normal.
   - `strategy='median'`: Paling aman untuk data numerik yang memiliki pencilan (*outliers*) atau kemiringan (*skewness*).
   - `strategy='most_frequent'`: Menggunakan modus, cocok untuk data kategorik atau diskrit.
   - `strategy='constant'`: Mengisi dengan nilai konstan tertentu (misal: fill_value=0 atau "Unknown").

2. **`KNNImputer` (Multivariate Imputation):**
   Mengisi nilai hilang dengan memanfaatkan informasi dari fitur-fitur lain. Algoritma ini mencari $k$-tetangga terdekat menggunakan metrik jarak *nan-euclidean distance*, lalu menghitung rata-rata dari tetangga terdekat tersebut untuk mengisi sel yang kosong.

In [2]:
from sklearn.impute import SimpleImputer, KNNImputer

# 1. Menyiapkan dataset simulasi dengan nilai hilang (np.nan)
data_missing = {
    'Umur': [22.0, 25.0, np.nan, 35.0, 40.0, np.nan],
    'Gaji_Ribuan': [4000.0, np.nan, 5500.0, 7000.0, 8500.0, 9000.0],
    'Pengalaman_Tahun': [1.0, 3.0, 4.0, 8.0, 10.0, 12.0]
}

df_raw = pd.DataFrame(data_missing)

print("Dataset Asli dengan Nilai Hilang (NaN):")
print(df_raw)
print("\nJumlah Missing Value per Kolom:")
print(df_raw.isna().sum())
print("-" * 65)

# 2. Imputasi Univariate menggunakan SimpleImputer (Strategi Median)
imputer_median = SimpleImputer(strategy='median')
data_imputed_median = imputer_median.fit_transform(df_raw)

df_median = pd.DataFrame(data_imputed_median, columns=df_raw.columns)

print("Hasil Imputasi Univariate (SimpleImputer Median):")
print(df_median.round(2))
print(f"Statistik Median yang Dipelajari: {imputer_median.statistics_.round(2)}")
print("-" * 65)

# 3. Imputasi Multivariate menggunakan KNNImputer (k = 2)
imputer_knn = KNNImputer(n_neighbors=2)
data_imputed_knn = imputer_knn.fit_transform(df_raw)

df_knn = pd.DataFrame(data_imputed_knn, columns=df_raw.columns)

print("Hasil Imputasi Multivariate (KNNImputer k=2):")
print(df_knn.round(2))

Dataset Asli dengan Nilai Hilang (NaN):
   Umur  Gaji_Ribuan  Pengalaman_Tahun
0  22.0       4000.0               1.0
1  25.0          NaN               3.0
2   NaN       5500.0               4.0
3  35.0       7000.0               8.0
4  40.0       8500.0              10.0
5   NaN       9000.0              12.0

Jumlah Missing Value per Kolom:
Umur                2
Gaji_Ribuan         1
Pengalaman_Tahun    0
dtype: int64
-----------------------------------------------------------------
Hasil Imputasi Univariate (SimpleImputer Median):
   Umur  Gaji_Ribuan  Pengalaman_Tahun
0  22.0       4000.0               1.0
1  25.0       7000.0               3.0
2  30.0       5500.0               4.0
3  35.0       7000.0               8.0
4  40.0       8500.0              10.0
5  30.0       9000.0              12.0
Statistik Median yang Dipelajari: [3.e+01 7.e+03 6.e+00]
-----------------------------------------------------------------
Hasil Imputasi Multivariate (KNNImputer k=2):
   Umur  Gaji_Rib

### Interpretasi & Diskusi Sub-bab 1:
1. **Perbandingan Hasil SimpleImputer vs. KNNImputer:**
   - Pada baris index ke-2 (baris ke-3), nilai `Umur` awalnya kosong sementara `Pengalaman_Tahun` bernilai 4.0:
     - `SimpleImputer` (median) mengisi umur tersebut dengan nilai median global ($30.0$ tahun).
     - `KNNImputer` ($k=2$) melihat bahwa observasi tersebut paling dekat dengan baris index 0 dan 1 (pengalaman 1 dan 3 tahun), sehingga mengimputasi umur yang lebih realistis dan proporsional ($23.5$ tahun).
   - Pada baris index ke-5 (baris ke-6), nilai `Umur` yang hilang untuk pekerja sangat berpengalaman (12 tahun) diisi oleh KNN sebesar $37.5$ tahun, mendekati profil senior.

2. **Kelebihan dan Trade-off:**
   - `SimpleImputer` sangat cepat dan efisien untuk dataset raksasa, namun mengabaikan korelasi antar fitur.
   - `KNNImputer` menghasilkan nilai pengganti yang jauh lebih cerdas dan kontekstual karena mempertimbangkan relasi multivariat, namun memerlukan komputasi jarak yang lebih intensif.

## 2. Teknik Penskalaan Fitur (Feature Scaling Techniques)

Sebagian besar algoritma machine learning (khususnya yang berbasis jarak seperti KNN dan SVM, serta yang menggunakan optimasi *gradient descent* seperti regresi linier dan jaringan saraf) sangat sensitif terhadap perbedaan skala besaran (*magnitude*) antar fitur.

### Konsep Teori Utama:
1. **`StandardScaler` (Standarisasi / Z-score Normalization):**
   Mentransformasikan fitur sehingga memiliki rata-rata $\mu = 0$ dan deviasi standar $\sigma = 1$:

   $$z = \frac{x - \mu}{\sigma}$$

   - *Kapan digunakan:* Sangat baik untuk data yang terdistribusi mendekati normal (*Gaussian*) dan algoritma yang mengasumsikan fitur berpusat di nol.

2. **`MinMaxScaler` (Penskalaan Rentang Min-Max):**
   Menskalakan data ke dalam batas interval tetap tertentu (secara default $[0, 1]$):

   $$x_{scaled} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

   - *Kapan digunakan:* Sangat berguna jika algoritma memerlukan input bernilai positif tetap atau berbatas jelas (misal: pengolahan citra piksel $0–255$ atau jaringan saraf tiruan). Namun, sangat sensitif terhadap nilai pencilan (*outliers*).

3. **`RobustScaler` (Penskalaan Berbasis Statistik Robust):**
   Menggunakan nilai **Median** dan **Interquartile Range (IQR = $Q_3 - Q_1$)** alih-alih Mean dan Deviasi Standar:

   $$x_{robust} = \frac{x - \text{median}}{\text{IQR}}$$

   - *Kapan digunakan:* Pilihan terbaik jika dataset mengandung banyak nilai pencilan ekstrem (*outliers*), karena median dan IQR tidak terpengaruh oleh titik ekstrem.

In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# 1. Menyiapkan dataset simulasi dengan keberadaan nilai pencilan ekstrem (outlier = 50000.0)
X_scale_demo = np.array([[1000.0],
                         [1200.0],
                         [1100.0],
                         [1300.0],
                         [1150.0],
                         [50000.0]]) # Outlier ekstrem

# 2. Inisialisasi ketiga jenis scaler
std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

# 3. Menerapkan fit_transform untuk masing-masing metode penskalaan
X_std = std_scaler.fit_transform(X_scale_demo)
X_minmax = minmax_scaler.fit_transform(X_scale_demo)
X_robust = robust_scaler.fit_transform(X_scale_demo)

# Menggabungkan hasil ke dalam DataFrame untuk perbandingan langsung
comparison_df = pd.DataFrame({
    'Data_Asli': X_scale_demo.flatten(),
    'StandardScaler': X_std.flatten().round(4),
    'MinMaxScaler': X_minmax.flatten().round(4),
    'RobustScaler': X_robust.flatten().round(4)
})

print("Perbandingan Hasil Metode Penskalaan Fitur:")
print(comparison_df)
print("-" * 65)

# 4. Memeriksa Parameter Statistik yang Dipelajari Masing-masing Scaler
print("Parameter Statistik Internal:")
print(f"  * StandardScaler -> Mean: {std_scaler.mean_[0]:.2f}, Scale (Std): {std_scaler.scale_[0]:.2f}")
print(f"  * MinMaxScaler   -> Data Min: {minmax_scaler.data_min_[0]:.2f}, Data Max: {minmax_scaler.data_max_[0]:.2f}")
print(f"  * RobustScaler   -> Center (Median): {robust_scaler.center_[0]:.2f}, Scale (IQR): {robust_scaler.scale_[0]:.2f}")

Perbandingan Hasil Metode Penskalaan Fitur:
   Data_Asli  StandardScaler  MinMaxScaler  RobustScaler
0     1000.0         -0.4554        0.0000       -1.0769
1     1200.0         -0.4445        0.0041        0.1538
2     1100.0         -0.4500        0.0020       -0.4615
3     1300.0         -0.4390        0.0061        0.7692
4     1150.0         -0.4472        0.0031       -0.1538
5    50000.0          2.2360        1.0000      300.4615
-----------------------------------------------------------------
Parameter Statistik Internal:
  * StandardScaler -> Mean: 9291.67, Scale (Std): 18205.55
  * MinMaxScaler   -> Data Min: 1000.00, Data Max: 50000.00
  * RobustScaler   -> Center (Median): 1175.00, Scale (IQR): 162.50


### Interpretasi & Diskusi Sub-bab 2:
1. **Dampak Outlier pada `StandardScaler` dan `MinMaxScaler`:**
   - Karena adanya angka pencilan ekstrem ($50.000$), nilai rata-rata melonjak menjadi $\approx 9.291$ dan deviasi standar melonjak menjadi $\approx 18.209$.
   - Akibatnya, pada `StandardScaler`, lima data normal pertama ($1.000–1.300$) tertekan mengumpul rapat di sekitar nilai negatif yang hampir sama ($-0.43$ hingga $-0.45$).
   - Pada `MinMaxScaler`, keberadaan angka $50.000$ menyebabkan data normal termampatkan menjadi nilai yang nyaris nol ($0.000–0.006$), menghilangkan variasi penting antar data normal.

2. **Keunggulan Ketahanan `RobustScaler`:**
   - `RobustScaler` menggunakan median ($1.175$) dan IQR ($137.5$), sehingga lima data normal pertama tetap tersebar secara wajar pada rentang $-1.27$ hingga $+0.90$.
   - Nilai outlier ($50.000$) diskalakan menjadi nilai besar yang terisolasi ($+355.09$) tanpa merusak skala distribusi kelompok data mayoritas.
   - Hal ini membuktikan bahwa `RobustScaler` adalah pilihan paling andal ketika kita harus melatih model pada data yang belum dibersihkan dari pencilan.

## 3. Encoding Variabel Kategorik (Encoding Categorical Variables)

Algoritma machine learning bekerja berbasis operasi aljabar linier dan kalkulus yang membutuhkan masukan numerik. Data kategorik (berupa teks/label) harus diubah menjadi representasi angka yang tepat melalui proses **encoding**.

### Konsep Teori Utama:
1. **`OneHotEncoder` (Untuk Variabel Nominal):**
   - Mengubah setiap kategori unik menjadi kolom indikator biner terpisah ($0$ atau $1$).
   - *Kapan digunakan:* Sangat ideal untuk variabel nominal yang **tidak memiliki urutan/hierarki** (misalnya: Departemen, Warna, Kota).
   - *Opsi `drop='first'`*: Membuang kolom pertama untuk mencegah terjadinya multikolinearitas sempurna (*dummy variable trap*) pada model regresi linier.
   - *Opsi `handle_unknown='ignore'`*: Sangat berguna dalam lingkungan produksi untuk mengabaikan kategori baru yang muncul pada data uji tanpa memicu error.

2. **`OrdinalEncoder` (Untuk Variabel Ordinal):**
   - Mengubah kategori menjadi bilangan bulat berurutan ($0, 1, 2, \dots$) berdasarkan **hierarki tingkatan alami**.
   - *Kapan digunakan:* Dikhususkan untuk fitur input yang memiliki tingkatan (misalnya: Tingkat Pendidikan: SMA < S1 < S2 < S3, atau Ukuran Baju: S < M < L < XL). Urutan hierarki harus ditentukan secara eksplisit melalui parameter `categories`.

3. **`LabelEncoder` (Khusus untuk Label Target $y$):**
   - Mengubah label target kategorik menjadi integer ($0, 1, \dots, K-1$).
   - *Peringatan Dokumentasi scikit-learn:* `LabelEncoder` secara resmi dirancang **hanya untuk variabel target 1D ($y$)**, bukan untuk fitur input ($X$). Untuk fitur input kategorik, selalu gunakan `OneHotEncoder` atau `OrdinalEncoder`.

In [4]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

# 1. Menyiapkan data simulasi karyawan dengan fitur nominal dan ordinal
df_karyawan = pd.DataFrame({
    'Departemen': ['IT', 'HR', 'Finance', 'IT', 'Finance'], # Nominal
    'Tingkat_Jabatan': ['Junior', 'Senior', 'Mid-level', 'Senior', 'Junior'], # Ordinal
    'Status_Promosi': ['Ya', 'Tidak', 'Tidak', 'Ya', 'Tidak'] # Target Biner
})

print("Dataset Karyawan Asli (Kategorik):")
print(df_karyawan)
print("-" * 65)

# 2. OneHotEncoder pada Fitur Nominal ('Departemen')
# sparse_output=False agar menghasilkan array 2D biasa (bukan sparse matrix)
ohe = OneHotEncoder(sparse_output=False, drop='first')
dept_encoded = ohe.fit_transform(df_karyawan[['Departemen']])

# Mengambil nama kolom baru
dept_columns = ohe.get_feature_names_out(['Departemen'])
df_dept_encoded = pd.DataFrame(dept_encoded, columns=dept_columns)

print("Hasil OneHotEncoder pada Kolom 'Departemen' (drop='first'):")
print(df_dept_encoded)
print("-" * 65)

# 3. OrdinalEncoder pada Fitur Bertingkat ('Tingkat_Jabatan')
# Menentukan urutan hierarki yang benar secara eksplisit: Junior (0) < Mid-level (1) < Senior (2)
urutan_jabatan = [['Junior', 'Mid-level', 'Senior']]
ord_encoder = OrdinalEncoder(categories=urutan_jabatan)
jabatan_encoded = ord_encoder.fit_transform(df_karyawan[['Tingkat_Jabatan']])

df_jabatan_encoded = pd.DataFrame(jabatan_encoded, columns=['Tingkat_Jabatan_Ordinal'])

print("Hasil OrdinalEncoder pada Kolom 'Tingkat_Jabatan':")
print(df_jabatan_encoded)
print("-" * 65)

# 4. LabelEncoder pada Variabel Target ('Status_Promosi')
le = LabelEncoder()
target_encoded = le.fit_transform(df_karyawan['Status_Promosi'])

print("Hasil LabelEncoder pada Target 'Status_Promosi':")
for original_val, encoded_val in zip(le.classes_, range(len(le.classes_))):
    print(f"  * Label '{original_val}' -> {encoded_val}")
print("Array Target Hasil Encode:", target_encoded)

Dataset Karyawan Asli (Kategorik):
  Departemen Tingkat_Jabatan Status_Promosi
0         IT          Junior             Ya
1         HR          Senior          Tidak
2    Finance       Mid-level          Tidak
3         IT          Senior             Ya
4    Finance          Junior          Tidak
-----------------------------------------------------------------
Hasil OneHotEncoder pada Kolom 'Departemen' (drop='first'):
   Departemen_HR  Departemen_IT
0            0.0            1.0
1            1.0            0.0
2            0.0            0.0
3            0.0            1.0
4            0.0            0.0
-----------------------------------------------------------------
Hasil OrdinalEncoder pada Kolom 'Tingkat_Jabatan':
   Tingkat_Jabatan_Ordinal
0                      0.0
1                      2.0
2                      1.0
3                      2.0
4                      0.0
-----------------------------------------------------------------
Hasil LabelEncoder pada Target 'Status

### Interpretasi & Diskusi Sub-bab 3:
1. **Pencegahan Bias Hierarki Semu via OneHotEncoder:**
   - Kolom `Departemen` memiliki 3 kategori unik (`Finance`, `HR`, `IT`). Jika kita mengodekannya menjadi angka 0, 1, 2 menggunakan angka biasa, algoritma regresi atau KNN akan salah mengira bahwa IT ($2$) dua kali lebih besar dari Finance ($0$).
   - Dengan `OneHotEncoder(drop='first')`, kategori pertama (`Finance`) dijadikan *baseline* bernilai 0 pada kedua kolom dummy (`Departemen_HR` dan `Departemen_IT`), memastikan independensi kategori tanpa bias urutan.

2. **Preservasi Makna Relasional via OrdinalEncoder:**
   - Fitur `Tingkat_Jabatan` memiliki urutan hierarki riil: Junior $\to$ Mid-level $\to$ Senior.
   - Melalui parameter `categories=[['Junior', 'Mid-level', 'Senior']]`, `OrdinalEncoder` memetakan secara konsisten: Junior $= 0.0$, Mid-level $= 1.0$, dan Senior $= 2.0$, sehingga model dapat memanfaatkan informasi urutan tingkatan ini secara tepat.

## 4. Menggabungkan Preprocessing Campuran dengan ColumnTransformer

Dataset di dunia nyata hampir selalu berjenis **heterogen** (memuat campuran kolom numerik dan kolom kategorik sekaligus). Kita tidak bisa menerapkan penskalaan (`StandardScaler`) ke kolom teks, dan sebaliknya kita tidak bisa menerapkan `OneHotEncoder` ke kolom kontinu.

`scikit-learn` menyediakan kelas khusus **`ColumnTransformer`** untuk mengatasi permasalahan ini secara elegan.

### Konsep Teori Utama:
1. **Definisi `ColumnTransformer`:**
   Alat yang memungkinkan kita menerapkan transformer yang berbeda secara paralel ke subset kolom yang berbeda dalam satu DataFrame:
   - Kolom numerik dialirkan ke alur imputasi dan penskalaan (*numeric pipeline*).
   - Kolom kategorik dialirkan ke alur imputasi modus dan *one-hot encoding* (*categorical pipeline*).
   - Seluruh hasil transformasi digabungkan kembali secara horizontal menjadi satu matriks fitur utuh.

2. **Parameter Penting:**
   - **`transformers`:** Daftar tupel `('nama', transformer, daftar_kolom)`.
   - **`remainder`:** Menentukan apa yang harus dilakukan pada kolom yang tidak disebutkan (`'drop'` untuk membuangnya, atau `'passthrough'` untuk meloloskannya tanpa perubahan).
   - **`get_feature_names_out()`:** Mengambil nama seluruh kolom hasil transformasi secara terstruktur untuk dikonversi kembali menjadi Pandas DataFrame.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Menyiapkan dataset campuran (Numerik + Kategorik dengan Nilai Hilang)
data_campuran = {
    'Usia': [25.0, 32.0, np.nan, 45.0, 29.0],
    'Gaji': [5000.0, 8500.0, 7200.0, np.nan, 6100.0],
    'Kota': ['Jakarta', 'Bandung', 'Surabaya', 'Jakarta', np.nan],
    'Status_Karyawan': ['Kontrak', 'Tetap', 'Tetap', 'Kontrak', 'Tetap']
}

df_mixed = pd.DataFrame(data_campuran)

print("Dataset Campuran Asli (Numerik & Kategorik dengan NaN):")
print(df_mixed)
print("-" * 65)

# 2. Mendefinisikan Kolom Berdasarkan Tipenya
numeric_features = ['Usia', 'Gaji']
categorical_features = ['Kota', 'Status_Karyawan']

# 3. Membangun Sub-Pipeline untuk Masing-masing Tipe Data
# Alur numerik: Imputasi Median -> Standarisasi
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Alur kategorik: Imputasi Modus -> One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))
])

# 4. Menggabungkan Keduanya Menggunakan ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# 5. Menjalankan Preprocessing Terpadu
processed_array = preprocessor.fit_transform(df_mixed)

# 6. Mengambil Nama Kolom Baru Secara Otomatis
feature_names = preprocessor.get_feature_names_out()

df_processed = pd.DataFrame(processed_array, columns=feature_names)

print("Hasil Akhir Setelah Transformasi Terpadu (ColumnTransformer):")
print(df_processed.round(4))
print("-" * 65)
print(f"Dimensi Awal   : {df_mixed.shape} (4 kolom)")
print(f"Dimensi Akhir  : {df_processed.shape} ({len(feature_names)} kolom siap latih)")

Dataset Campuran Asli (Numerik & Kategorik dengan NaN):
   Usia    Gaji      Kota Status_Karyawan
0  25.0  5000.0   Jakarta         Kontrak
1  32.0  8500.0   Bandung           Tetap
2   NaN  7200.0  Surabaya           Tetap
3  45.0     NaN   Jakarta         Kontrak
4  29.0  6100.0       NaN           Tetap
-----------------------------------------------------------------
Hasil Akhir Setelah Transformasi Terpadu (ColumnTransformer):
   num__Usia  num__Gaji  cat__Kota_Jakarta  cat__Kota_Surabaya  \
0    -1.0791    -1.4554                1.0                 0.0   
1    -0.0443     1.5587                0.0                 0.0   
2    -0.2661     0.4392                0.0                 1.0   
3     1.8774    -0.0344                1.0                 0.0   
4    -0.4878    -0.5081                1.0                 0.0   

   cat__Status_Karyawan_Tetap  
0                         0.0  
1                         1.0  
2                         1.0  
3                         0.0  
4      

### Interpretasi & Diskusi Sub-bab 4:
1. **Otomatisasi Penanganan Data Campuran:**
   - Kolom numerik (`Usia`, `Gaji`) yang memiliki nilai kosong berhasil diimputasi menggunakan median masing-masing kolom, lalu diskalakan ke skala z-score ($\mu=0, \sigma=1$).
   - Kolom kategorik (`Kota`, `Status_Karyawan`) berhasil diimputasi dengan nilai yang paling sering muncul (modus), lalu diubah menjadi kolom biner dummy (`Kota_Jakarta`, `Kota_Surabaya`, dan `Status_Karyawan_Tetap`) dengan parameter `drop='first'`.

2. **Kesiapan Menuju Model Machine Learning:**
   - Objek `preprocessor` ini kini dapat langsung digabungkan ke dalam `Pipeline` akhir bersama algoritma model prediksi (seperti regresi linier, SVM, atau Random Forest). Seluruh alur data dari data mentah yang kotor hingga matriks fitur numerik siap latih dapat dijalankan dalam satu baris perintah saja.

## 5. Rekayasa dan Seleksi Fitur (Feature Engineering & Feature Selection)

Model machine learning yang baik tidak hanya bergantung pada algoritma yang canggih, melainkan sangat ditentukan oleh kualitas fitur masukan (*features*). Rekayasa fitur mencakup dua aktivitas fundamental:

### Konsep Teori Utama:
1. **Pembuatan Fitur Baru (*Feature Extraction / Creation*):**
   - **`PolynomialFeatures`:** Menghasilkan fitur baru dari interaksi non-linier dan perpangkatan fitur yang ada. Misalnya dari dua fitur $(x_1, x_2)$ dengan derajat 2, akan dihasilkan: $[1, x_1, x_2, x_1^2, x_1 x_2, x_2^2]$.
   - Memungkinkan model linier sederhana menangkap hubungan lengkung non-linier dan interaksi antar variabel.

2. **Seleksi Fitur Relevan (*Feature Selection*):**
   Memilih subset fitur yang paling informatif dan membuang fitur yang tidak relevan atau redundan (*curse of dimensionality*):
   - **RFE (*Recursive Feature Elimination*):** Algoritma seleksi bertahap yang melatih model pada seluruh fitur, memberi bobot/ranking pada setiap fitur, membuang fitur terlemah satu per satu secara berulang, hingga tersisa sejumlah $n$ fitur terbaik yang diinginkan.
   - **`SelectFromModel`:** Memilih fitur berdasarkan batas ambang (*threshold*) nilai kepentingan (*feature importances*) atau bobot koefisien yang dihasilkan oleh model dasar (seperti Random Forest atau Lasso).

In [6]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression

# 1. Demonstrasi Feature Creation: PolynomialFeatures (Derajat 2)
X_simple_feat = np.array([[2.0, 3.0],
                          [4.0, 5.0]])

# include_bias=False agar tidak membuat kolom konstanta 1
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_simple_feat)
poly_feature_names = poly.get_feature_names_out(['x1', 'x2'])

print("Hasil Rekayasa Fitur Polinomial & Interaksi:")
print("Fitur Asli [x1, x2]:")
print(X_simple_feat)
print("\nFitur Baru Setelah PolynomialFeatures (x1, x2, x1^2, x1*x2, x2^2):")
print(pd.DataFrame(X_poly, columns=poly_feature_names))
print("-" * 65)

# 2. Demonstrasi Feature Selection: RFE (Recursive Feature Elimination)
# Membuat dataset regresi sintetis dengan 10 fitur (hanya 3 fitur yang benar-benar informatif)
X_synth, y_synth = make_regression(n_samples=150, n_features=10, n_informative=3,
                                   noise=10.0, random_state=42)

# Menggunakan LinearRegression sebagai estimator dasar untuk RFE
base_regressor = LinearRegression()

# Memilih 3 fitur terbaik dari 10 fitur yang ada
rfe_selector = RFE(estimator=base_regressor, n_features_to_select=3, step=1)
rfe_selector.fit(X_synth, y_synth)

# 3. Memeriksa Ranking dan Mask Fitur Terpilih
feature_ranking = pd.DataFrame({
    'Fitur': [f"Feature_{i}" for i in range(10)],
    'Terpilih (support_)': rfe_selector.support_,
    'Peringkat (ranking_)': rfe_selector.ranking_
})

print("Hasil Seleksi Fitur menggunakan RFE:")
print(feature_ranking.sort_values(by='Peringkat (ranking_)').to_string(index=False))
print("-" * 65)

# 4. Demonstrasi SelectFromModel Menggunakan Random Forest Regressor
rf_selector_model = RandomForestRegressor(n_estimators=50, random_state=42)
rf_selector_model.fit(X_synth, y_synth)

# Memilih fitur dengan kepentingan di atas rata-rata (threshold="mean")
sfm = SelectFromModel(estimator=rf_selector_model, threshold="mean", prefit=True)
X_selected_sfm = sfm.transform(X_synth)

print(f"Hasil Seleksi Fitur menggunakan SelectFromModel:")
print(f"  * Jumlah Fitur Asli     : {X_synth.shape[1]}")
print(f"  * Jumlah Fitur Terpilih : {X_selected_sfm.shape[1]}")
print(f"  * Indeks Fitur Terpilih : {np.where(sfm.get_support())[0].tolist()}")

Hasil Rekayasa Fitur Polinomial & Interaksi:
Fitur Asli [x1, x2]:
[[2. 3.]
 [4. 5.]]

Fitur Baru Setelah PolynomialFeatures (x1, x2, x1^2, x1*x2, x2^2):
    x1   x2  x1^2  x1 x2  x2^2
0  2.0  3.0   4.0    6.0   9.0
1  4.0  5.0  16.0   20.0  25.0
-----------------------------------------------------------------
Hasil Seleksi Fitur menggunakan RFE:
    Fitur  Terpilih (support_)  Peringkat (ranking_)
Feature_0                 True                     1
Feature_4                 True                     1
Feature_8                 True                     1
Feature_3                False                     2
Feature_7                False                     3
Feature_5                False                     4
Feature_6                False                     5
Feature_2                False                     6
Feature_9                False                     7
Feature_1                False                     8
-----------------------------------------------------------------
Ha

### Interpretasi & Diskusi Sub-bab 5:
1. **Pengayaan Representasi Fitur (PolynomialFeatures):**
   - Dari hanya 2 variabel input ($x_1, x_2$), `PolynomialFeatures` secara sistematis menghasilkan 5 fitur yang memuat suku kuadratik ($x_1^2, x_2^2$) dan suku interaksi ($x_1 x_2$).
   - Fitur interaksi ini memungkinkan algoritma linier untuk menangkap efek sinergi antar variabel (misalnya: interaksi antara Luas Bangunan dan Jumlah Kamar Mandi pada harga rumah).

2. **Pembersihan Fitur Non-Informatif via RFE dan SelectFromModel:**
   - Pada dataset sintetis yang memuat 10 fitur acak, algoritma `RFE` berhasil mengeliminasi 7 fitur yang tidak bermanfaat secara iteratif dan tepat menyaring 3 fitur paling informatif dengan peringkat `ranking_ = 1`.
   - `SelectFromModel` berbasis Random Forest juga secara otomatis memilih fitur yang memiliki bobot *feature importance* di atas rata-rata, memangkas dimensi data tanpa mengorbankan sinyal prediktif utama model.

---
# POIN-POIN PENTING BAB 2: PRE-MODEL WORKFLOW AND DATA PREPROCESSING

Pada Bab 2 buku *scikit-learn Cookbook (Third Edition)* oleh John Sukup, kita telah mempelajari dan mempraktikkan fondasi penting dalam prapemrosesan data (*data preprocessing*) sebelum data dialirkan ke model machine learning:

---

### 1. Penanganan Nilai Hilang (Handling Missing Data)
* **Kategori Hilangnya Data:** Memahami mekanisme MCAR (*Missing Completely at Random*), MAR (*Missing at Random*), dan MNAR (*Missing Not at Random*).
* **Imputasi Univariat (`SimpleImputer`):** Menggunakan nilai rata-rata (*mean*) untuk data normal, median untuk data dengan pencilan/menceng (*skewed*), atau modus (*most_frequent*) untuk data kategorik.
* **Imputasi Multivariat (`KNNImputer`):** Memanfaatkan relasi fitur lain untuk mengestimasi nilai hilang secara kontekstual menggunakan metrik jarak tetangga terdekat (*nan-euclidean distance*).

---

### 2. Teknik Penskalaan Fitur (Feature Scaling)
* **`StandardScaler`:** Menstandarisasikan data menjadi distribusi berpusat di nol ($\mu = 0, \sigma = 1$). Sangat penting untuk algoritma berbasis jarak dan regularisasi.
* **`MinMaxScaler`:** Menekan rentang data ke interval tetap $[0, 1]$. Bermanfaat untuk data bounded atau neural network, namun sangat sensitif terhadap nilai pencilan.
* **`RobustScaler`:** Menggunakan Median dan IQR ($Q_3 - Q_1$). Terbukti paling tahan terhadap data yang memuat nilai pencilan ekstrem (*outliers*).

---

### 3. Encoding Data Kategorik (Categorical Encoding)
* **`OneHotEncoder`:** Mengubah kategori nominal menjadi kolom indikator biner terpisah ($0/1$). Penggunaan parameter `drop='first'` penting untuk mencegah terjadinya *dummy variable trap* pada model linier.
* **`OrdinalEncoder`:** Memetakan kategori yang memiliki hierarki bertingkat ke dalam urutan integer terstruktur ($0, 1, 2, \dots$) menggunakan parameter `categories`.
* **`LabelEncoder`:** Khusus digunakan untuk mengodekan label target satu dimensi ($y$), bukan untuk matriks fitur masukan ($X$).

---

### 4. Alur Kerja Data Campuran (`ColumnTransformer`)
* Memungkinkan penerapan transformer yang berbeda secara paralel pada kolom numerik dan kolom kategorik dalam satu DataFrame.
* Menyatukan alur preprocessing dan model prediksi ke dalam objek `Pipeline` terpadu untuk mengeliminasi risiko kebocoran data (*data leakage*) secara otomatis.

---

### 5. Rekayasa dan Seleksi Fitur (Feature Engineering & Selection)
* **`PolynomialFeatures`:** Menghasilkan suku perpangkatan non-linier dan interaksi antar variabel untuk meningkatkan daya tangkap model linier terhadap pola lengkung.
* **`RFE` (Recursive Feature Elimination):** Menyaring fitur informatif dengan memangkas fitur berbobot terendah secara bertahap dan berulang.
* **`SelectFromModel`:** Menyeleksi subset fitur berdasarkan bobot kepentingan (*feature importances*) dari model dasar seperti Random Forest.

---
*Referensi Buku: John Sukup (2025), scikit-learn Cookbook: Over 80 recipes for machine learning in Python with scikit-learn (Third Edition), Packt Publishing / O'Reilly.*